# Routing Optimization — Exploration

The vehicle routing problem with time windows (VRPTW) on the Solomon
100-customer benchmark: a fleet leaves a depot, serves every customer
inside a promised time window, respects vehicle capacity, and minimizes
total distance.

This notebook demonstrates the project's essence in three steps:

1. load one prepared benchmark instance through the package's own config
   and parser;
2. read the headline solver-ladder comparison from the persisted
   `reports/metrics.json`;
3. reproduce one key result in under a second — the HPO-tuned ALNS plan
   for C101 — and check it against the persisted run and the published
   best-known reference.

## How to run

From the project directory (`projects/06-routing-optimization`):

```bash
uv sync --all-packages
```

then run the cells top to bottom (e.g. `uv run jupyter nbconvert --to
notebook --execute --inplace notebooks/exploration.ipynb`, or interactively
in Jupyter/VS Code).

Everything here reads persisted artifacts — instance files under
`data/raw/` and pipeline outputs under `reports/` — which are produced by
`uv run dvc repro` (run from the repo root or this project dir). Nothing
is re-downloaded and no step needs network access.

## 1. Load a prepared benchmark instance

C101 is the classic clustered, tight-window instance: customers form tight
geographic clusters, but their narrow time windows dictate the visit order
more than the map does. The package's config points at `data/raw/` and the
parser turns the Solomon text format into typed `Problem`/`Customer`
models.

In [ ]:
from routing_optimization.config import get_vrp_config
from routing_optimization.parser import parse_solomon_file

cfg = get_vrp_config()
problem = parse_solomon_file(cfg.raw_dir / "C101.txt", name="C101")

customers = problem.customers[1:]  # index 0 is the depot
windows = [c.due_time - c.ready_time for c in customers]
print(
    f"{problem.name}: {len(customers)} customers, capacity {problem.vehicle_capacity:g}, "
    f"fleet up to {problem.max_vehicles} vehicles"
)
print(
    f"time windows {min(windows):.0f}-{max(windows):.0f} min wide, "
    f"demands {min(c.demand for c in customers):g}-{max(c.demand for c in customers):g}"
)

## 2. The headline comparison — the solver ladder

The pipeline ran six seeded, deterministic solvers on all 56 Solomon
instances and persisted per-solver aggregates to `reports/metrics.json`
(via `uv run dvc repro`). The ladder reads bottom-up: feasibility is cheap,
constraint-aware construction is the first big lever, local polish helps a
little, and the metaheuristic (ALNS) closes the rest.

In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path("reports/metrics.json").read_text(encoding="utf-8"))
summary = metrics["summary"]

ladder = ["randomized", "nearest_neighbor", "clarke_wright", "two_opt", "two_opt_tuned", "alns"]
print(f"{'solver':<16}{'mean gap %':>12}{'mean vehicles':>16}{'mean runtime':>14}")
for solver in ladder:
    s = summary[solver]
    print(
        f"{solver:<16}{s['mean_gap_pct']:>11.2f} %{s['mean_n_vehicles']:>15.2f}"
        f"{s['mean_runtime_s']:>11.3f} s"
    )

instances = metrics["instances"]
wins = sum(1 for inst in instances.values() if inst["best_solver"] == "alns")
print(f"\nALNS is the best solver on {wins}/{len(instances)} instances (seed 42 run)")

## 3. Reproduce one key result — ALNS on C101

The same code the pipeline runs, in seconds: Clarke-Wright savings
construction, the HPO-tuned 2-opt polish (budget 50, best-improvement,
2-opt + 2-opt*), then ALNS destroy-and-repair from the tuned plan with
the HPO-winning knobs (segment 100, reaction 0.5). Every plan passes the
hard validator — each customer served exactly once, capacity and every
time window respected — before it is compared against the published
best-known distance.

In [ ]:
from routing_optimization.evaluate import gap_pct, load_best_known
from routing_optimization.solver import alns, clarke_wright_savings, two_opt

best_known = load_best_known(cfg.best_known_path, cfg.best_known_supplement_path)

base = clarke_wright_savings(problem)
tuned = two_opt(problem, base, max_iterations=50, acceptance="best", tail_exchange=True)
plan = alns(
    problem,
    start=tuned,
    seed=cfg.seed,
    max_iterations=cfg.alns_max_iterations,
    segment_length=100,  # HPO winner (params.yaml)
    reaction_factor=0.5,
    init_weights="uniform",
    destroy_q=cfg.alns_destroy_q,
)
plan.validate_against(problem)  # hard gates: once-only, capacity, time windows

persisted = instances["C101"]["solvers"]["alns"]
print(f"clarke_wright : {base.total_distance:9.2f}  ({base.n_vehicles} vehicles)")
print(f"two_opt tuned : {tuned.total_distance:9.2f}  ({tuned.n_vehicles} vehicles)")
print(f"alns (re-run) : {plan.total_distance:9.2f}  ({plan.n_vehicles} vehicles)")
print(f"alns persisted: {persisted['distance']:9.2f}  ({persisted['n_vehicles']} vehicles)")
print(
    f"best-known reference {best_known['C101']:.2f} -> "
    f"gap {gap_pct(plan.total_distance, best_known['C101']):.2f} %"
)

## Where to go next

- Case study: [/projects/06-routing-optimization](/projects/06-routing-optimization)
  — the full narrative, results tables and figures.
- [DESIGN.md](../DESIGN.md) — architecture decisions: solver contract,
  seeded determinism, the evaluation gates.
- Code: `src/routing_optimization/` — `parser.py` (Solomon format),
  `solver.py` (the six solvers), `evaluate.py` (benchmark + figures),
  `tuning.py` (seeded HPO), `cli.py` (`download` / `solve` / `report`).

To regenerate every artifact end to end: `uv run dvc repro` (downloads the
sha256-pinned Solomon archive on first run, then solves all 56 instances
including both HPO searches in about 90 s).